In [ ]:
import pandas as pd

df = pd.read_csv("orders_hw41.csv", parse_dates=["placed_at"])

df["revenue"] = df["quantity"] * df["price_each"]

df["month"] = df["placed_at"].dt.to_period("M").astype(str)

paid = df[df["status"] == "paid"]

monthly = (paid.groupby(["country", "month"], as_index=False)
 .agg(revenue=("revenue", "sum"), orders=
("order_id", "count")))

top = (monthly.sort_values(["country", "revenue"], ascending=
[True, False])
 .groupby("country", as_index=False).head(1))
top.to_csv("best_month_per_country.csv", index=False)

print(top)

In [ ]:
import polars as pl
df = pl.read_csv("orders_hw41.csv")

df = df.with_columns([
    (pl.col("quantity") * pl.col("price_each")).alias("revenue"),

    pl.col("placed_at")
      .str.strptime(pl.Date, "%Y-%m-%d", strict=False)
      .dt.strftime("%Y-%m")
      .alias("month")
])

paid = (df.filter(pl.col("status") == "paid"))

monthly = (paid.group_by("country", "month")).agg((pl.col("revenue").sum()).alias("revenue"), pl.col("order_id").count().alias("orders"))

top = (monthly.sort("country", "revenue", descending=[False, True])
 .group_by("country")
 .head(1)
 .sort("country"))

top.write_csv("best_month_per_country_polars.csv")

print(top)

In [ ]:
# API differences hit during Pandas → Polars migration:
#
# 1. Pandas reads dates directly with read_csv(..., parse_dates=["placed_at"]); in Polars I parsed the string column explicitly with str.strptime(pl.Date, "%Y-%m-%d").
# 2. Pandas adds columns by assignment, e.g. df["revenue"] = ...; Polars uses df.with_columns([...]).
# 3. Pandas references columns as df["quantity"]; Polars uses expression syntax with pl.col("quantity").
# 4. Pandas filters rows with df[df["status"] == "paid"]; Polars uses df.filter(pl.col("status") == "paid").
# 5. Pandas uses .dt.to_period("M").astype(str) for month extraction; Polars uses .dt.strftime("%Y-%m").
# 6. Pandas groupby uses groupby(...).agg(revenue=("revenue", "sum"), orders=("order_id", "count")); Polars uses group_by(...).agg(...) with explicit aliases.
# 7. Pandas sort_values uses ascending=[True, False]; Polars sort uses descending=[False, True].
# 8. Pandas writes CSV with to_csv(..., index=False); Polars writes CSV with write_csv(...) and has no index by default.
# 9. Polars group_by/head may not preserve final row order like Pandas, so I added a final sort("country") to stabilize output.